# Greenwashing Detection Pipeline
**MSIN0166 — Data Engineering Group Assignment | UCL School of Management 2025/26**

## Pipeline Overview
This notebook builds an end-to-end data engineering pipeline to detect potential greenwashing 
among FTSE 100 companies by comparing self-reported ESG claims against independent data sources.

### Data Sources
| # | Source | Type | Storage |
|---|--------|------|---------|
| 1 | Company sustainability websites | Web scraping | MongoDB + Parquet |
| 2 | The Guardian API | News API | MongoDB + Parquet |
| 3 | Reddit Public JSON | Social media | MongoDB + Parquet |
| 4 | Our World in Data CO2 | Open dataset | Parquet |
| 5 | NetworkX Graph | Relationship model | GEXF |

### Processing Stack
- **Apache Spark** — distributed joins, aggregations, derived columns
- **DuckDB** — analytical SQL queries on Parquet files
- **MongoDB** — NoSQL storage for unstructured text

---
## Section 0: Setup & Imports
Load all libraries and API keys from the `.env` file. 
Create output directories for raw and processed data.

In [1]:
# Install dependencies (run once)
#pip install requests beautifulsoup4 pandas pyarrow pymongo pyspark duckdb newsapi-python python-dotenv

In [2]:
import os, re, json, time, requests
import pandas as pd
import pymongo
import pyarrow.parquet as pq
import duckdb
import networkx as nx
from datetime import datetime
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, when, lit, count, avg, 
                                    round as spark_round, sum as spark_sum,
                                    dense_rank, desc)
from pyspark.sql.window import Window

load_dotenv()
NEWS_API_KEY            = os.getenv("NEWS_API_KEY")
GUARDIAN_API_KEY        = os.getenv("GUARDIAN_API_KEY")
MONGO_URI               = os.getenv("MONGO_URI", "mongodb://localhost:27017/")
COMPANIES_HOUSE_API_KEY = os.getenv("COMPANIES_HOUSE_API_KEY", "")

os.makedirs("data/raw",       exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Clear lineage log at start of fresh run
with open("data/lineage_log.jsonl", "w") as f:
    pass

print("Imports successful")
print(f"Guardian API key : {'✅' if GUARDIAN_API_KEY else '⚠️ Missing'}")
print(f"MongoDB URI      : {MONGO_URI}")

c:\Users\VEDANT M BHATIA\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\VEDANT M BHATIA\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Imports successful
Guardian API key : ⚠️ Missing
MongoDB URI      : mongodb://localhost:27017/


### Lineage Logger
Every data collection step writes a structured entry to `data/lineage_log.jsonl`.
This creates a full, reproducible audit trail — a core data engineering best practice.
Each entry records: source name, URL, timestamp, record count, output path, and transformations applied.

In [3]:
LINEAGE_FILE = "data/lineage_log.jsonl"

def log_lineage(source_name, source_url, record_count, output_path, transformations=None):
    entry = {
        "source":          source_name,
        "url":             source_url,
        "extracted_at":    datetime.now().isoformat(),
        "record_count":    int(record_count),
        "output_path":     output_path,
        "transformations": transformations or []
    }
    with open(LINEAGE_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")
    print(f"[LINEAGE] {source_name} → {record_count} records → {output_path}")

print("Lineage logger ready")

Lineage logger ready


---
## Section 1: Web Scraping — Company ESG Claims

We scrape the public sustainability pages of 10 FTSE 100 companies to extract 
self-reported ESG claims. This is our primary dataset — the "claims" side of the 
greenwashing analysis.

**What we extract:**
- Net-zero target year (e.g. "net zero by 2050")
- Emissions reduction % claimed (e.g. "reduce emissions by 50%")
- Third-party certifications (SBTi, CDP, TCFD, ISO14001, RE100)
- Raw text stored in MongoDB for potential NLP analysis

**Why scraping is necessary:** This data does not exist in any structured API.
It only lives as free text on company websites.

**Known limitation:** Several companies (GSK, AstraZeneca, M&S) use JavaScript 
rendering which prevents static HTML scraping. This is documented as a real-world 
data engineering challenge — it would require Selenium/Playwright to resolve, which 
is outside scope for this pipeline.

**Ethics:** 2-second delay between requests, standard browser User-Agent, 
no login walls circumvented, robots.txt respected.

In [4]:
import io
import pyarrow as pa
import pyarrow.parquet as pq

COMPANIES = [
    {
        
        "company_id": "BP", "company_name": "BP", "sector": "Energy",
        "urls": [
            "https://www.bp.com/en/global/corporate/sustainability.html",
            "https://www.bp.com/en/global/corporate/sustainability/getting-to-net-zero.html",
            "https://en.wikipedia.org/wiki/BP",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "BARC", "company_name": "Barclays", "sector": "Finance",
        "urls": [
            "https://home.barclays/sustainability/",
            "https://home.barclays/sustainability/esg-resource-hub/",
            "https://home.barclays/investor-relations/reports-and-events/annual-reports/",
            "https://en.wikipedia.org/wiki/Barclays",
            "https://home.barclays/news/press-releases/",
            "https://home.barclays/insights/sustainability-insights/",
            "https://web.archive.org/web/20231015/https://home.barclays/sustainability/net-zero/",
            "https://web.archive.org/web/20231015/https://home.barclays/sustainability/our-strategy/",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "LLOY", "company_name": "Lloyds", "sector": "Finance",
        "urls": [
            "https://www.lloydsbankinggroup.com/who-we-are/responsible-business.html",
            "https://www.lloydsbankinggroup.com/investors/annual-report.html",
            "https://en.wikipedia.org/wiki/Lloyds_Banking_Group",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "RDSB", "company_name": "Rio Tinto", "sector": "Mining",
        "urls": [
            "https://www.riotinto.com/sustainability",
            "https://www.riotinto.com/sustainability/climate-change",
            "https://en.wikipedia.org/wiki/Rio_Tinto_(corporation)",
            "https://www.riotinto.com/sustainability/climate-change/paris-agreement"
        ]
    },
    {
        
        "company_id": "GSK", "company_name": "GSK", "sector": "Healthcare",
        "urls": [
            "https://www.gsk.com/en-gb/responsibility/environment/",
            "https://www.gsk.com/en-gb/responsibility/environment/net-zero/",
            "https://www.gsk.com/en-gb/responsibility/our-esg-reporting/",
            "https://www.gsk.com/en-gb/responsibility/environment/environmental-data/",
            "https://web.archive.org/web/20231015/https://www.gsk.com/en-gb/responsibility/environment/",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "AZN", "company_name": "AstraZeneca", "sector": "Healthcare",
        "urls": [
            "https://en.wikipedia.org/wiki/AstraZeneca",
            "https://web.archive.org/web/20231015/https://www.astrazeneca.com/sustainability.html",
            "https://web.archive.org/web/20231015/https://www.astrazeneca.com/sustainability/ambition-zero-carbon.html",
            "https://web.archive.org/web/20231015/https://www.astrazeneca.com/sustainability/esg-reporting.html",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "MKS", "company_name": "Marks & Spencer", "sector": "Retail",
        "urls": [
            "https://en.wikipedia.org/wiki/Marks_%26_Spencer",
            "https://web.archive.org/web/20231015/https://corporate.marksandspencer.com/sustainability/planet",
            "https://web.archive.org/web/20231015/https://corporate.marksandspencer.com/sustainability/planet/net-zero",
            "https://web.archive.org/web/20231015/https://corporate.marksandspencer.com/sustainability/planet/energy-and-carbon",
            "https://corporate.marksandspencer.com/investors/results-reports-and-presentations",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "VOD", "company_name": "Vodafone", "sector": "Telecom",
        "urls": [
            "https://en.wikipedia.org/wiki/Vodafone",
            "https://www.vodafone.com/sustainability",
            "https://web.archive.org/web/20231015/https://www.vodafone.com/sustainability/reporting-and-policies",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "NATG", "company_name": "National Grid", "sector": "Utilities",
        "urls": [
            "https://www.nationalgrid.com/responsibility",
            "https://web.archive.org/web/20231015/https://www.nationalgrid.com/responsibility/environment",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "SBRY", "company_name": "Sainsburys", "sector": "Retail",
        "urls": [
            "https://www.about.sainsburys.co.uk/sustainability",
            "https://www.about.sainsburys.co.uk/sustainability/plan-for-better/net-zero",
            "https://www.about.sainsburys.co.uk/sustainability/plan-for-better/our-planet",
            "https://en.wikipedia.org/wiki/Sainsbury%27s",
            "https://www.about.sainsburys.co.uk/news",
            "https://www.about.sainsburys.co.uk/sustainability/plan-for-better/net-zero/our-carbon-footprint",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    }
]

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

YEAR_PATTERNS = [
    r"net.zero\D{0,30}(20\d{2})",         r"(20\d{2})\D{0,30}net.zero",
    r"carbon.neutral\D{0,30}(20\d{2})",   r"(20\d{2})\D{0,30}carbon.neutral",
    r"zero.carbon\D{0,20}(20\d{2})",      r"net.zero.by.(20\d{2})",
    r"climate.neutral\D{0,30}(20\d{2})",  r"(20\d{2})\D{0,30}climate.neutral",
    r"achieve.net.zero\D{0,20}(20\d{2})", r"zero.emissions\D{0,20}(20\d{2})"
]

PCT_PATTERNS = [
    r"reduc\w+\D{0,15}(\d{1,3})\s*%",   r"(\d{1,3})\s*%\D{0,15}reduc",
    r"cut\D{0,15}(\d{1,3})\s*%",         r"(\d{1,3})\s*%\D{0,20}emission",
    r"emissions\D{0,20}(\d{1,3})\s*%",   r"decarboni\w+\D{0,15}(\d{1,3})\s*%",
    r"(\d{1,3})\s*%\D{0,20}carbon",      r"lower\D{0,15}(\d{1,3})\s*%",
    r"(\d{1,3})\s*% absolute reduction",  r"scope.1.and.2\D{0,20}(\d{1,3})\s*%"
]

CERT_KEYWORDS = {
    "SBTi":     ["science based targets", "sbti", "science-based targets"],
    "CDP":      ["cdp", "carbon disclosure project"],
    "TCFD":     ["tcfd", "task force on climate"],
    "ISO14001": ["iso 14001", "iso14001"],
    "RE100":    ["re100", "100% renewable energy"],
    "PAS2060":  ["pas2060", "pas 2060"]
}

def extract_from_text(text):
    net_zero_year, reduction_pct, certs = None, None, []

    for pat in YEAR_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            yr = int(m.group(1))
            
            if 2025 <= yr <= 2060:
                net_zero_year = yr
                break

    for pat in PCT_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            val = float(m.group(1))
            # Normalise to 0-100 scale:
            # Values like 0.5 or 8.0 are likely on 0-1 or 0-10 scale — scale up
            if 0 < val < 1:
                val = round(val * 100, 1)   # 0.5 → 50.0
            elif 1 <= val < 10:
                val = round(val * 10, 1)    # 8.0 → 80.0
            # Accept only meaningful headline claims (10-99% on 0-100 scale)
            if 10 <= val <= 99:
                reduction_pct = val; break

    for cert, kws in CERT_KEYWORDS.items():
        if any(kw in text.lower() for kw in kws):
            certs.append(cert)

    return net_zero_year, reduction_pct, certs

def scrape_single_url(url):
    try:
        time.sleep(2)
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        tags = soup.find_all(["p","h1","h2","h3","li","span","div"])
        paras = [t.get_text(strip=True) for t in tags if len(t.get_text(strip=True)) > 30]
        return " ".join(paras[:300])
    except:
        return None

def scrape_company_multi_url(company: dict) -> dict:
    result = {
        "company_id":            company["company_id"],
        "company_name":          company["company_name"],
        "sector":                company["sector"],
        "net_zero_target_year":  None,
        "reduction_pct_claimed": None,
        "certifications":        set(),
        "raw_text":              "",
        "urls_scraped":          [],
        "urls_successful":       0,
        "source_url":            company["urls"][0],
        "scraped_at":            datetime.now().isoformat()
    }

    for url in company["urls"]:
        if (result["net_zero_target_year"] is not None and
            result["reduction_pct_claimed"] is not None and
            len(result["certifications"]) >= 2):
            print(f"      ↳ All fields found — stopping early ✅")
            break

        label = ("wayback→" + url.split("https://")[-1].split("/")[0]
                 if "web.archive.org" in url
                 else "/".join(url.split("/")[2:4]))
        print(f"      ↳ {label}...", end=" ")
        text = scrape_single_url(url)

        if text:
            year, pct, certs = extract_from_text(text)
            result["urls_successful"] += 1
            result["raw_text"]        += " " + text[:800]
            result["urls_scraped"].append(url)

            if result["net_zero_target_year"] is None and year:
                result["net_zero_target_year"] = year
                print(f"year={year} ✅", end=" ")
            if result["reduction_pct_claimed"] is None and pct:
                result["reduction_pct_claimed"] = pct
                print(f"pct={pct}% ✅", end=" ")
            if certs:
                new = set(certs) - result["certifications"]
                if new:
                    result["certifications"].update(new)
                    print(f"certs={list(new)} ✅", end=" ")
            print()
        else:
            print("⚠️ JS/blocked")

    result["certifications"] = ",".join(sorted(result["certifications"])) \
                                if result["certifications"] else None
    result["raw_text"]       = result["raw_text"].strip()[:5000]
    return result

print(f" Total URLs: {sum(len(c['urls']) for c in COMPANIES)}")

 Total URLs: 52


In [5]:
print("Starting multi-URL web scraping...\n")
print("=" * 60)

scraped_results = []
for company in COMPANIES:
    print(f"\n  📋 {company['company_name']} ({len(company['urls'])} URLs):")
    result = scrape_company_multi_url(company)
    scraped_results.append(result)
    print(f"  ✅ RESULT → year={result['net_zero_target_year']} | "
          f"pct={result['reduction_pct_claimed']} | "
          f"certs={result['certifications']} | "
          f"URLs ok: {result['urls_successful']}/{len(company['urls'])}")

claims_df = pd.DataFrame(scraped_results)

print(f"\n{'=' * 60}")
print(f"SCRAPING COMPLETE")
print(f"  Net zero year : {claims_df['net_zero_target_year'].notna().sum()}/10")
print(f"  Reduction %   : {claims_df['reduction_pct_claimed'].notna().sum()}/10")
print(f"  Certifications: {claims_df['certifications'].notna().sum()}/10")
print(f"  Avg URLs ok   : {claims_df['urls_successful'].mean():.1f}/4")
print(f"{'=' * 60}")


claims_df[["company_name","net_zero_target_year","reduction_pct_claimed",
           "certifications","urls_successful"]]

Starting multi-URL web scraping...


  📋 BP (4 URLs):
      ↳ www.bp.com/en... year=2050 ✅ 
      ↳ www.bp.com/en... pct=20.0% ✅ 
      ↳ en.wikipedia.org/wiki... 
      ↳ sciencebasedtargets.org/companies-taking-action... certs=['SBTi', 'CDP'] ✅ 
  ✅ RESULT → year=2050 | pct=20.0 | certs=CDP,SBTi | URLs ok: 4/4

  📋 Barclays (9 URLs):
      ↳ home.barclays/sustainability... 
      ↳ home.barclays/sustainability... 
      ↳ home.barclays/investor-relations... certs=['TCFD'] ✅ 
      ↳ en.wikipedia.org/wiki... 
      ↳ home.barclays/news... 
      ↳ home.barclays/insights... pct=15.0% ✅ 
      ↳ wayback→home.barclays... ⚠️ JS/blocked
      ↳ wayback→home.barclays... ⚠️ JS/blocked
      ↳ sciencebasedtargets.org/companies-taking-action... year=2050 ✅ certs=['SBTi', 'CDP'] ✅ 
  ✅ RESULT → year=2050 | pct=15.0 | certs=CDP,SBTi,TCFD | URLs ok: 7/9

  📋 Lloyds (4 URLs):
      ↳ www.lloydsbankinggroup.com/who-we-are... year=2050 ✅ pct=50.0% ✅ 
      ↳ www.lloydsbankinggroup.com/investors... 


,company_name,net_zero_target_year,reduction_pct_claimed,certifications,urls_successful
0,BP,2050,20.0,"CDP,SBTi",4
1,Barclays,2050,15.0,"CDP,SBTi,TCFD",7
2,Lloyds,2050,50.0,"CDP,SBTi",4
3,Rio Tinto,2050,15.0,SBTi,4
4,GSK,2030,80.0,"RE100,SBTi,TCFD",5
5,AstraZeneca,2050,30.0,"CDP,SBTi",5
6,Marks & Spencer,2050,90.0,"CDP,SBTi",5
7,Vodafone,2050,84.0,"CDP,SBTi",3
8,National Grid,2050,60.0,"CDP,SBTi",3
9,Sainsburys,2050,80.0,"CDP,SBTi",7


### Store Scraped Data
Raw text goes to MongoDB (unstructured, variable length — ideal for NoSQL document store).
Structured fields go to Parquet (columnar format, optimised for Spark and DuckDB queries).
This dual-storage pattern is intentional — different storage engines serve different query patterns.

In [6]:

# ── MongoDB: store raw text (NoSQL — ideal for unstructured variable-length content)
try:
    mongo_client = pymongo.MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    mongo_client.server_info()
    db = mongo_client["greenwashing"]
    db["company_claims_raw"].drop()
    db["company_claims_raw"].insert_many(scraped_results)
    print(f"✅ MongoDB: {len(scraped_results)} records → greenwashing.company_claims_raw")
except Exception as e:
    print(f"⚠️ MongoDB unavailable: {e}")

# ── Parquet: structured fields only, saved via BytesIO (Windows permission safe)
os.makedirs("data/raw", exist_ok=True)
claims_structured = claims_df.drop(columns=["raw_text","urls_scraped"], errors="ignore")
buffer = io.BytesIO()
pq.write_table(pa.Table.from_pandas(claims_structured, preserve_index=False), buffer)
with open("data/raw/company_claims.parquet", "wb") as f:
    f.write(buffer.getvalue())
print(f"✅ Parquet: saved → data/raw/company_claims.parquet")
print(f"   Columns: {list(claims_structured.columns)}")

log_lineage(
    source_name  = "FTSE100 Multi-URL Web Scraping (4 URLs per company)",
    source_url   = "Sustainability pages + Climate pages + Wikipedia + Press releases",
    record_count = len(claims_df),
    output_path  = "data/raw/company_claims.parquet",
    transformations = [
        "4 URLs per company: sustainability, climate subpage, Wikipedia, press releases",
        "8 regex patterns for net-zero year extraction",
        "8 regex patterns for reduction % extraction",
        "6 certification types matched (SBTi, CDP, TCFD, ISO14001, RE100, PAS2060)",
        "Findings merged per company — all URLs feed one record",
        "Raw text stored in MongoDB, structured fields in Parquet"
    ]
)
print(f"✅ Saved → data/raw/company_claims.parquet ({os.path.getsize('data/raw/company_claims.parquet'):,} bytes)")
claims_df[["company_name","net_zero_target_year","reduction_pct_claimed","certifications","urls_successful"]]


✅ MongoDB: 10 records → greenwashing.company_claims_raw
✅ Parquet: saved → data/raw/company_claims.parquet
   Columns: ['company_id', 'company_name', 'sector', 'net_zero_target_year', 'reduction_pct_claimed', 'certifications', 'urls_successful', 'source_url', 'scraped_at']
[LINEAGE] FTSE100 Multi-URL Web Scraping (4 URLs per company) → 10 records → data/raw/company_claims.parquet
✅ Saved → data/raw/company_claims.parquet (7,644 bytes)


,company_name,net_zero_target_year,reduction_pct_claimed,certifications,urls_successful
0,BP,2050,20.0,"CDP,SBTi",4
1,Barclays,2050,15.0,"CDP,SBTi,TCFD",7
2,Lloyds,2050,50.0,"CDP,SBTi",4
3,Rio Tinto,2050,15.0,SBTi,4
4,GSK,2030,80.0,"RE100,SBTi,TCFD",5
5,AstraZeneca,2050,30.0,"CDP,SBTi",5
6,Marks & Spencer,2050,90.0,"CDP,SBTi",5
7,Vodafone,2050,84.0,"CDP,SBTi",3
8,National Grid,2050,60.0,"CDP,SBTi",3
9,Sainsburys,2050,80.0,"CDP,SBTi",7


## Section 2: The Guardian API — News Articles

The Guardian provides a free open API (api-key=test) requiring no registration.
We use it to collect journalism covering each company's ESG and climate record.

**Why The Guardian specifically?**
The Guardian is the UK's leading investigative journalism outlet for climate and 
corporate accountability reporting. For UK-listed FTSE 100 companies, it provides 
higher quality and more relevant coverage than general news aggregators.

**Search strategy:** 3 queries per company targeting different angles:
1. `{company} greenwashing` — direct greenwashing allegations
2. `{company} emissions climate` — climate performance reporting  
3. `{company} ESG sustainability` — ESG coverage

Articles are deduplicated by URL to avoid counting the same article twice.

In [7]:
def fetch_guardian_articles(company_name: str) -> list:
    url = "https://content.guardianapis.com/search"
    records = []
    queries = [f"{company_name} greenwashing",
               f"{company_name} emissions climate",
               f"{company_name} ESG sustainability"]

    for query in queries:
        try:
            time.sleep(1)
            resp = requests.get(url, params={
                "q": query, "page-size": 10,
                "show-fields": "headline,trailText,byline,wordcount",
                "api-key": "test"
            }, timeout=15)
            resp.raise_for_status()
            for art in resp.json().get("response", {}).get("results", []):
                records.append({
                    "company_name": company_name,
                    "headline":     art.get("fields", {}).get("headline", art.get("webTitle")),
                    "snippet":      art.get("fields", {}).get("trailText"),
                    "byline":       art.get("fields", {}).get("byline"),
                    "published_at": art.get("webPublicationDate"),
                    "section":      art.get("sectionName"),
                    "url":          art.get("webUrl"),
                    "api_source":   "Guardian Open API",
                    "extracted_at": datetime.now().isoformat()
                })
        except Exception as e:
            print(f"  ⚠️ Guardian ({query[:25]}): {e}")

    # Deduplicate by URL
    seen, unique = set(), []
    for r in records:
        if r["url"] not in seen:
            seen.add(r["url"]); unique.append(r)
    return unique

print("Fetching Guardian articles...\n")
all_articles = []
for company in COMPANIES:
    print(f"  {company['company_name']}...", end=" ")
    arts = fetch_guardian_articles(company["company_name"])
    all_articles.extend(arts)
    print(f"{len(arts)} articles")
    time.sleep(1)

news_df = pd.DataFrame(all_articles)
news_df["published_at"] = pd.to_datetime(news_df["published_at"], errors="coerce")
news_df["company_name"] = news_df["company_name"].str.strip()

# MongoDB
try:
    db["news_articles_raw"].drop()
    db["news_articles_raw"].insert_many(all_articles)
    print(f"\n✅ MongoDB: {len(all_articles)} articles → greenwashing.news_articles_raw")
except: pass

news_df.to_parquet("data/raw/news_articles.parquet", index=False)
log_lineage("The Guardian Open API", "content.guardianapis.com/search",
            len(news_df), "data/raw/news_articles.parquet",
            transformations=["3 queries per company", "Deduplicated by URL",
                             "Stored in MongoDB and Parquet"])

print(f"\nTotal articles: {len(news_df)}")
news_df[["company_name","headline","section","published_at"]].head(10)

Fetching Guardian articles...

  BP... 25 articles
  Barclays... 27 articles
  Lloyds... 24 articles
  Rio Tinto... 22 articles
  GSK... 23 articles
  AstraZeneca... 24 articles
  Marks & Spencer... 18 articles
  Vodafone... 25 articles
  National Grid... 26 articles
  Sainsburys... 30 articles

✅ MongoDB: 244 articles → greenwashing.news_articles_raw
[LINEAGE] The Guardian Open API → 244 records → data/raw/news_articles.parquet

Total articles: 244


,company_name,headline,section,published_at
0,BP,Claims that AI can help fix climate dismissed ...,Technology,2026-02-17 05:00:47+00:00
1,BP,UniSuper accused of greenwashing after reducin...,Environment,2026-02-20 01:38:45+00:00
2,BP,Bunnings accused of ‘greenwashing’ timber amid...,Australia news,2026-01-26 14:00:05+00:00
3,BP,‘Landmark’ greenwashing case against Australia...,Australia news,2026-02-17 04:41:31+00:00
4,BP,"Greenwashing, illegality and false claims: 13 ...",Environment,2025-12-31 17:00:15+00:00
5,BP,Morning Mail: Bunnings accused of ‘greenwashin...,Australia news,2026-01-26 19:46:05+00:00
6,BP,BP halts share buy-backs as annual profits slide,Business,2026-02-10 10:31:46+00:00
7,BP,BP accused of ‘insidious’ influence on UK educ...,Business,2026-01-16 06:00:01+00:00
8,BP,Honeymoon period for new BP boss won’t last long,Business,2026-02-10 16:57:02+00:00
9,BP,BP faces calls for new strategy to end period ...,Business,2026-02-08 12:38:58+00:00


## Section 3: Reddit Public JSON API — Community Sentiment

Reddit exposes a public `.json` endpoint on every search page, requiring no 
authentication. We use this to capture community-level sentiment — real public 
opinion about these companies' environmental records.

**Why Reddit?**
Reddit discussions represent organic public sentiment from informed communities.
Subreddits like r/investing and r/environment contain detailed analysis from users 
who follow these companies closely. High upvote scores and comment counts indicate 
posts the community found significant.

**Search strategy:**
- Global Reddit search (all subreddits) — 25 posts per company
- 8 targeted subreddits confirmed to be publicly accessible
- 4-second delay between requests to avoid rate limiting (HTTP 429)
- 5-second delay between companies to reset rate limit window
- Deduplicated by post_id to avoid counting reposts

In [8]:
def fetch_reddit_posts(company_name: str) -> list:
    records = []
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    def parse_posts(posts, source):
        results = []
        for post in posts:
            d = post.get("data", {})
            results.append({
                "company_name": company_name,
                "post_id":      d.get("id"),
                "title":        d.get("title"),
                "text":         d.get("selftext", "")[:500],
                "score":        d.get("score"),
                "upvote_ratio": d.get("upvote_ratio"),
                "num_comments": d.get("num_comments"),
                "subreddit":    d.get("subreddit"),
                "url":          f"https://reddit.com{d.get('permalink','')}",
                "created_utc":  datetime.utcfromtimestamp(d.get("created_utc",0)).isoformat(),
                "api_source":   "Reddit Public JSON",
                "extracted_at": datetime.now().isoformat()
            })
        return results

    # Global Reddit-wide search
    try:
        time.sleep(3)
        resp = requests.get("https://www.reddit.com/search.json", headers=headers,
            params={"q": f"{company_name} greenwashing OR emissions OR ESG OR \"net zero\" OR sustainability OR climate",
                    "sort": "relevance", "limit": 25, "t": "all"}, timeout=15)
        resp.raise_for_status()
        posts = resp.json().get("data",{}).get("children",[])
        records.extend(parse_posts(posts, "global"))
        print(f"global:{len(posts)}", end=" ")
    except: print("global:skip", end=" ")

    # Targeted subreddits
    for sub in ["investing","environment","sustainability","CorporateMalfeasance",
                "anticonsumption","collapse","worldnews","Economics"]:
        try:
            time.sleep(4)
            resp = requests.get(f"https://www.reddit.com/r/{sub}/search.json",
                headers=headers,
                params={"q": f"{company_name} greenwashing OR emissions OR ESG OR climate",
                        "restrict_sr":"true","sort":"relevance","limit":10,"t":"all"},
                timeout=15)
            resp.raise_for_status()
            posts = resp.json().get("data",{}).get("children",[])
            records.extend(parse_posts(posts, sub))
            print(f"r/{sub}:{len(posts)}", end=" ")
        except: print(f"r/{sub}:skip", end=" ")

    # Deduplicate by post_id
    seen, unique = set(), []
    for r in records:
        if r["post_id"] not in seen:
            seen.add(r["post_id"]); unique.append(r)
    return unique

print("Fetching Reddit posts...\n")
all_reddit = []
for company in COMPANIES:
    print(f"\n  {company['company_name']}: ", end="")
    posts = fetch_reddit_posts(company["company_name"])
    all_reddit.extend(posts)
    print(f"  → {len(posts)} unique posts")
    time.sleep(5)

reddit_df = pd.DataFrame(all_reddit)
reddit_df["score"]        = pd.to_numeric(reddit_df["score"],        errors="coerce").fillna(0).astype(int)
reddit_df["num_comments"] = pd.to_numeric(reddit_df["num_comments"], errors="coerce").fillna(0).astype(int)
reddit_df["upvote_ratio"] = pd.to_numeric(reddit_df["upvote_ratio"], errors="coerce").fillna(0.0)
reddit_df["company_name"] = reddit_df["company_name"].str.strip()
reddit_df = reddit_df[reddit_df["title"].notna()]

# MongoDB
try:
    db["reddit_posts_raw"].drop()
    db["reddit_posts_raw"].insert_many(reddit_df.to_dict("records"))
    print(f"\n✅ MongoDB: {len(reddit_df)} posts → greenwashing.reddit_posts_raw")
except: pass

reddit_df.to_parquet("data/raw/reddit_posts.parquet", index=False)
log_lineage("Reddit Public JSON API",
            "reddit.com (global + r/investing, r/environment, r/sustainability, r/worldnews)",
            len(reddit_df), "data/raw/reddit_posts.parquet",
            transformations=["Global search + 8 subreddits", "Deduplicated by post_id",
                             "Stored in MongoDB and Parquet"])

print(f"\nTotal posts: {len(reddit_df)}")
print(reddit_df["company_name"].value_counts().to_string())

Fetching Reddit posts...


  BP: global:25 r/investing:10 r/environment:10 r/sustainability:10 r/CorporateMalfeasance:6 r/anticonsumption:10 r/collapse:10 r/worldnews:10 r/Economics:10   → 100 unique posts

  Barclays: global:25 r/investing:10 r/environment:10 r/sustainability:4 r/CorporateMalfeasance:3 r/anticonsumption:2 r/collapse:6 r/worldnews:10 r/Economics:10   → 80 unique posts

  Lloyds: global:25 r/investing:10 r/environment:10 r/sustainability:2 r/CorporateMalfeasance:1 r/anticonsumption:2 r/collapse:10 r/worldnews:0 r/Economics:10   → 70 unique posts

  Rio Tinto: global:25 r/investing:10 r/environment:10 r/sustainability:1 r/CorporateMalfeasance:1 r/anticonsumption:0 r/collapse:2 r/worldnews:10 r/Economics:6   → 60 unique posts

  GSK: global:25 r/investing:10 r/environment:0 r/sustainability:0 r/CorporateMalfeasance:0 r/anticonsumption:1 r/collapse:0 r/worldnews:10 r/Economics:2   → 47 unique posts

  AstraZeneca: global:25 r/investing:10 r/environment:4 r/sustainability:0

## Section 4: Our World in Data — Verified UK CO2 Emissions

Our World in Data (OWID) publishes a comprehensive, peer-reviewed CO2 dataset 
maintained by climate researchers. It is freely available as a CSV on GitHub.

**Why this source?**
This provides the "ground truth" emissions data to compare against company claims.
If a company claims to have reduced emissions by 50% but national data shows the 
sector's emissions are flat or rising, that is a potential greenwashing signal.

**What we extract:** UK-only records from 2000 onwards including total CO2, 
CO2 per capita, total GHG, and fuel-type breakdowns (coal, oil, gas).

In [9]:
def fetch_owid_co2():
    url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
    try:
        df = pd.read_csv(url)
        uk = df[(df["country"] == "United Kingdom") & (df["year"] >= 2000)].copy()
        uk = uk[["country","year","co2","co2_per_capita","ghg_per_capita",
                 "energy_per_gdp","total_ghg","coal_co2","oil_co2","gas_co2"]].reset_index(drop=True)
        uk["source"]       = "Our World in Data CO2 Dataset"
        uk["extracted_at"] = datetime.now().isoformat()
        return uk
    except Exception as e:
        print(f"  ❌ OWID error: {e}")
        return pd.DataFrame()

print("Fetching OWID CO2 data...")
owid_df = fetch_owid_co2()
owid_df.to_parquet("data/raw/owid_co2.parquet", index=False)

log_lineage("Our World in Data — UK CO2 Dataset",
            "github.com/owid/co2-data",
            len(owid_df), "data/raw/owid_co2.parquet",
            transformations=["Filtered: United Kingdom only",
                             "Filtered: year >= 2000",
                             "Selected 10 emissions columns"])

print(f"✅ {len(owid_df)} records | {owid_df['year'].min()}–{owid_df['year'].max()}")
owid_df.tail(8)

Fetching OWID CO2 data...
[LINEAGE] Our World in Data — UK CO2 Dataset → 25 records → data/raw/owid_co2.parquet
✅ 25 records | 2000–2024


,country,year,co2,co2_per_capita,ghg_per_capita,energy_per_gdp,total_ghg,coal_co2,oil_co2,gas_co2,source,extracted_at
17,United Kingdom,2017,387.367,5.838,7.202,0.886,477.853,39.129,174.851,161.697,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179
18,United Kingdom,2018,379.730,5.689,7.120,0.870,475.272,33.280,173.087,161.485,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179
19,United Kingdom,2019,364.753,5.435,6.765,0.836,453.993,24.513,168.996,159.321,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179
20,United Kingdom,2020,326.263,4.844,6.114,0.849,411.774,22.809,143.981,149.420,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179
21,United Kingdom,2021,342.366,5.059,6.292,0.793,425.768,23.758,149.626,158.503,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179
22,United Kingdom,2022,311.118,4.563,5.751,0.769,392.132,19.979,146.334,135.047,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179
23,United Kingdom,2023,307.826,4.482,5.629,NaN,386.611,17.914,154.684,125.766,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179
24,United Kingdom,2024,312.906,4.526,5.645,NaN,390.275,16.948,159.600,126.620,Our World in Data CO2 Dataset,2026-03-04T11:42:04.188179


## Section 5: Graph Database — Company Relationship Network

We model the relationships between companies, sectors, and certifications 
as a directed graph using NetworkX.

**Why a graph database?**
Graph databases excel at relationship queries that are inefficient or impossible 
in flat tables. For our pipeline, the graph answers questions like:
- Which certifications are shared across multiple sectors?
- Which companies are isolated (no certifications, no sector peers)?
- How connected is the ESG certification network?

**Graph structure:**
- **Nodes:** Companies (10), Sectors (5), Certifications (variable)
- **Edges:** company → sector (operates_in), company → certification (certified_by)
- **Node attributes:** company_id, net_zero_year, reduction_pct stored on each node

The graph is exported as GEXF format — the standard for graph databases, 
openable in Gephi for visual network analysis.

In [10]:
G = nx.DiGraph()

for _, row in claims_df.iterrows():
    company = row["company_name"]
    sector  = row["sector"]

    G.add_node(company, type="company",
               company_id    = row["company_id"],
               net_zero_year = str(row["net_zero_target_year"]),
               reduction_pct = str(row["reduction_pct_claimed"]))

    G.add_node(sector, type="sector")
    G.add_edge(company, sector, relationship="operates_in")

    if row["certifications"] and str(row["certifications"]) != "None":
        for cert in str(row["certifications"]).split(","):
            cert = cert.strip()
            if cert:
                G.add_node(cert, type="certification")
                G.add_edge(company, cert, relationship="certified_by")

nx.write_gexf(G, "data/processed/company_graph.gexf")

# Print summary
node_types = {}
for _, attrs in G.nodes(data=True):
    t = attrs.get("type","unknown")
    node_types[t] = node_types.get(t,0) + 1

print(f"✅ Graph database saved → data/processed/company_graph.gexf")
print(f"\n  Nodes: {G.number_of_nodes()}")
for t,c in node_types.items():
    print(f"    {t}: {c}")
print(f"\n  Edges: {G.number_of_edges()}")
for src, tgt, attrs in G.edges(data=True):
    print(f"    {src} → {tgt} [{attrs.get('relationship')}]")

log_lineage("NetworkX Graph — Company ESG Relationships",
            "data/raw/company_claims.parquet",
            G.number_of_nodes(), "data/processed/company_graph.gexf",
            transformations=["company→sector edges", "company→certification edges",
                             "Node attributes: net_zero_year, reduction_pct",
                             "Exported as GEXF"])

✅ Graph database saved → data/processed/company_graph.gexf

  Nodes: 21
    company: 10
    sector: 7
    certification: 4

  Edges: 31
    BP → Energy [operates_in]
    BP → CDP [certified_by]
    BP → SBTi [certified_by]
    Barclays → Finance [operates_in]
    Barclays → CDP [certified_by]
    Barclays → SBTi [certified_by]
    Barclays → TCFD [certified_by]
    Lloyds → Finance [operates_in]
    Lloyds → CDP [certified_by]
    Lloyds → SBTi [certified_by]
    Rio Tinto → Mining [operates_in]
    Rio Tinto → SBTi [certified_by]
    GSK → Healthcare [operates_in]
    GSK → RE100 [certified_by]
    GSK → SBTi [certified_by]
    GSK → TCFD [certified_by]
    AstraZeneca → Healthcare [operates_in]
    AstraZeneca → CDP [certified_by]
    AstraZeneca → SBTi [certified_by]
    Marks & Spencer → Retail [operates_in]
    Marks & Spencer → CDP [certified_by]
    Marks & Spencer → SBTi [certified_by]
    Vodafone → Telecom [operates_in]
    Vodafone → CDP [certified_by]
    Vodafone → SBTi [c

## Section 6: Build Social Signal Table

Before loading into Spark, we aggregate the Reddit and Guardian data 
into a single company-level signal table. This becomes one of the 
key join keys in the Spark pipeline.

**Social signal columns per company:**
- `reddit_post_count` — volume of community discussion
- `reddit_avg_score` — average upvotes (measures community significance)
- `reddit_total_comments` — total engagement depth
- `news_article_count` — volume of journalism coverage
- `total_media_signals` — combined media exposure score

High media signals combined with no certifications is our primary greenwashing flag.

In [11]:
reddit_agg = reddit_df.groupby("company_name").agg(
    reddit_post_count     = ("post_id",      "count"),
    reddit_avg_score      = ("score",        "mean"),
    reddit_total_comments = ("num_comments", "sum")
).reset_index()

news_agg = news_df.groupby("company_name").agg(
    news_article_count = ("headline", "count")
).reset_index()

social_signal = reddit_agg.merge(news_agg, on="company_name", how="outer").fillna(0)
social_signal["reddit_avg_score"]      = social_signal["reddit_avg_score"].round(0).astype(int)
social_signal["reddit_total_comments"] = social_signal["reddit_total_comments"].astype(int)
social_signal["reddit_post_count"]     = social_signal["reddit_post_count"].astype(int)
social_signal["news_article_count"]    = social_signal["news_article_count"].astype(int)
social_signal["total_media_signals"]   = (social_signal["reddit_post_count"] + 
                                           social_signal["news_article_count"])

social_signal.to_parquet("data/raw/social_signal.parquet", index=False)
log_lineage("Social Signal Table (Reddit + Guardian aggregated)",
            "data/raw/reddit_posts.parquet + data/raw/news_articles.parquet",
            len(social_signal), "data/raw/social_signal.parquet",
            transformations=["Reddit: post count, avg score, total comments per company",
                             "Guardian: article count per company",
                             "Outer merge into unified signal table"])

print("✅ Social signal table:")
print(social_signal[["company_name","reddit_post_count","reddit_avg_score",
                       "news_article_count","total_media_signals"]].to_string(index=False))


[LINEAGE] Social Signal Table (Reddit + Guardian aggregated) → 10 records → data/raw/social_signal.parquet
✅ Social signal table:
   company_name  reddit_post_count  reddit_avg_score  news_article_count  total_media_signals
    AstraZeneca                 60              6398                  24                   84
             BP                100              4616                  25                  125
       Barclays                 80              3449                  27                  107
            GSK                 47              1547                  23                   70
         Lloyds                 70              5958                  24                   94
Marks & Spencer                 43              1027                  18                   61
  National Grid                 76              2543                  26                  102
      Rio Tinto                 60              2568                  22                   82
     Sainsburys         

## Section 7: Apache Spark — Distributed Processing

Spark is our scalable processing layer. Even though our dataset is small (10 companies),
using Spark demonstrates that this pipeline can scale to thousands of companies 
without changing a single line of code.

**What Spark does in this pipeline:**
1. Loads all Parquet files as distributed DataFrames
2. Joins company claims ← social signal ← OWID CO2 data
3. Adds derived columns: risk flags, risk scores, claim credibility score
4. Runs Spark SQL analytical queries across the merged dataset
5. Saves the final processed dataset as Parquet

**Greenwashing Risk Score logic (0–4):**
| Signal | Points |
|--------|--------|
| Reduction claim > 30% with no certification | +2 |
| No third-party certifications at all | +1 |
| High media attention (>50 Reddit posts) | +1 |

In [12]:
import os

# Fix for Windows — must be set BEFORE creating SparkSession
os.environ["HADOOP_HOME"]      = "C:\\hadoop"
os.environ["hadoop.home.dir"]  = "C:\\hadoop"

# Use forward-slash absolute paths for Spark on Windows
import pathlib
base_path = str(pathlib.Path("data").resolve()).replace("\\", "/")

spark = SparkSession.builder \
    .appName("GreenwashingPipeline") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", f"{base_path}/spark-warehouse") \
    .config("spark.local.dir",         f"{base_path}/spark-temp") \
    .config("spark.hadoop.hadoop.tmp.dir", f"{base_path}/spark-temp") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} started")
print(f"✅ HADOOP_HOME = {os.environ['HADOOP_HOME']}")

✅ Spark 4.1.1 started
✅ HADOOP_HOME = C:\hadoop


### Step 7.1 — Load and Join All Sources
Load Parquet files and perform left joins so all 10 companies are retained 
even if some have no social media coverage.

In [13]:
# Load all sources
claims_sp = spark.read.parquet("data/raw/company_claims.parquet")
social_sp = spark.read.parquet("data/raw/social_signal.parquet")
owid_sp   = spark.read.parquet("data/raw/owid_co2.parquet")

print("Schema — Company Claims:")
claims_sp.printSchema()
print(f"Claims:  {claims_sp.count()} rows")
print(f"Social:  {social_sp.count()} rows")
print(f"OWID:    {owid_sp.count()} rows")

# Get most recent UK CO2 figure for context
latest_co2 = owid_sp.orderBy(desc("year")).limit(1) \
                     .select("year", "co2", "total_ghg", "co2_per_capita")
print("\nLatest UK CO2 data:")
latest_co2.show()

# Join claims with social signal
merged = claims_sp.join(social_sp, on="company_name", how="left") \
                  .fillna({"reddit_post_count":0, "reddit_avg_score":0,
                            "reddit_total_comments":0, "news_article_count":0,
                            "total_media_signals":0})

print(f"\nMerged dataset: {merged.count()} rows")
merged.show(truncate=False)


Schema — Company Claims:
root
 |-- company_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- net_zero_target_year: long (nullable = true)
 |-- reduction_pct_claimed: double (nullable = true)
 |-- certifications: string (nullable = true)
 |-- urls_successful: long (nullable = true)
 |-- source_url: string (nullable = true)
 |-- scraped_at: string (nullable = true)

Claims:  10 rows
Social:  10 rows
OWID:    25 rows

Latest UK CO2 data:
+----+-------+---------+--------------+
|year|    co2|total_ghg|co2_per_capita|
+----+-------+---------+--------------+
|2024|312.906|  390.275|         4.526|
+----+-------+---------+--------------+


Merged dataset: 10 rows
+---------------+----------+----------+--------------------+---------------------+---------------+---------------+-----------------------------------------------------------------------+--------------------------+-----------------+----------------+---------------------


### Step 7.2 — Add Derived Columns and Risk Score
We engineer five derived columns from the raw data. These turn raw signals 
into actionable greenwashing risk indicators.

In [14]:
merged = merged \
    .withColumn("has_net_zero_target",
                when(col("net_zero_target_year").isNotNull(), 1).otherwise(0)) \
    .withColumn("high_reduction_claim",
                when(col("reduction_pct_claimed") > 30, 1).otherwise(0)) \
    .withColumn("no_certification",
                when(col("certifications").isNull(), 1).otherwise(0)) \
    .withColumn("high_media_attention",
                when(col("total_media_signals") > 50, 1).otherwise(0)) \
    .withColumn("claim_without_cert",
                when((col("reduction_pct_claimed") > 30) & 
                     (col("certifications").isNull()), 1).otherwise(0)) \
    .withColumn("greenwashing_risk_score",
                # +2 for ambitious claim with no certification (strongest signal)
                (col("claim_without_cert") * 2) +
                # +1 for any missing certification
                col("no_certification") +
                # +1 for high media attention (scrutiny signal)
                col("high_media_attention")) \
    .withColumn("risk_category",
                when(col("greenwashing_risk_score") >= 3, "HIGH")
                .when(col("greenwashing_risk_score") == 2, "MEDIUM")
                .when(col("greenwashing_risk_score") == 1, "LOW")
                .otherwise("MINIMAL")) \
    .withColumn("processed_at", lit(datetime.now().isoformat()))

print("✅ Derived columns added:")
merged.select("company_name","has_net_zero_target","high_reduction_claim",
              "no_certification","high_media_attention","claim_without_cert",
              "greenwashing_risk_score","risk_category").show(truncate=False)

✅ Derived columns added:
+---------------+-------------------+--------------------+----------------+--------------------+------------------+-----------------------+-------------+
|company_name   |has_net_zero_target|high_reduction_claim|no_certification|high_media_attention|claim_without_cert|greenwashing_risk_score|risk_category|
+---------------+-------------------+--------------------+----------------+--------------------+------------------+-----------------------+-------------+
|BP             |1                  |0                   |0               |1                   |0                 |1                      |LOW          |
|Barclays       |1                  |0                   |0               |1                   |0                 |1                      |LOW          |
|Lloyds         |1                  |1                   |0               |1                   |0                 |1                      |LOW          |
|Rio Tinto      |1                  |0             

### Step 7.3 — Spark SQL Analytical Queries
We register the merged DataFrame as a SQL view and run structured queries.
Spark SQL allows us to express complex analytical logic in familiar SQL syntax 
while benefiting from Spark's distributed execution engine.

In [15]:
# Register as SQL views
merged.createOrReplaceTempView("greenwashing")
owid_sp.createOrReplaceTempView("uk_co2")

# ── Query A: Full risk ranking ──
print("=== GREENWASHING RISK RANKING ===")
spark.sql("""
    SELECT company_name, sector, net_zero_target_year,
           reduction_pct_claimed, certifications,
           reddit_post_count, news_article_count, total_media_signals,
           greenwashing_risk_score, risk_category
    FROM greenwashing
    ORDER BY greenwashing_risk_score DESC, total_media_signals DESC
""").show(truncate=False)

# ── Query B: Sector aggregation ──
print("=== SECTOR-LEVEL RISK ANALYSIS ===")
spark.sql("""
    SELECT sector,
           COUNT(*)                              AS company_count,
           ROUND(AVG(reduction_pct_claimed),1)   AS avg_reduction_claimed,
           ROUND(AVG(greenwashing_risk_score),2)  AS avg_risk_score,
           SUM(no_certification)                  AS uncertified_companies,
           SUM(claim_without_cert)                AS unverified_claims,
           SUM(total_media_signals)               AS total_sector_media
    FROM greenwashing
    GROUP BY sector
    ORDER BY avg_risk_score DESC
""").show(truncate=False)

# ── Query C: High risk companies ──
print("=== HIGH RISK COMPANIES (Score >= 2) ===")
spark.sql("""
    SELECT company_name, sector, reduction_pct_claimed,
           certifications, total_media_signals,
           greenwashing_risk_score, risk_category
    FROM greenwashing
    WHERE greenwashing_risk_score >= 2
    ORDER BY greenwashing_risk_score DESC
""").show(truncate=False)

# ── Query D: Media attention vs claims ──
print("=== MEDIA ATTENTION vs CLAIMS ===")
spark.sql("""
    SELECT company_name,
           COALESCE(CAST(reduction_pct_claimed AS STRING), 'No claim') AS reduction_claimed,
           COALESCE(certifications, 'None')                             AS certifications,
           reddit_post_count,
           news_article_count,
           total_media_signals,
           risk_category
    FROM greenwashing
    ORDER BY total_media_signals DESC
""").show(truncate=False)

# ── Query E: UK CO2 trend ──
print("=== UK NATIONAL CO2 TREND (2015-2024) ===")
spark.sql("""
    SELECT year,
           ROUND(co2, 2)             AS co2_MtCO2,
           ROUND(total_ghg, 2)       AS total_ghg,
           ROUND(co2_per_capita, 2)  AS co2_per_capita,
           ROUND(coal_co2, 2)        AS coal_co2,
           ROUND(oil_co2, 2)         AS oil_co2,
           ROUND(gas_co2, 2)         AS gas_co2
    FROM uk_co2
    WHERE year >= 2015
    ORDER BY year DESC
""").show(truncate=False)

=== GREENWASHING RISK RANKING ===
+---------------+----------+--------------------+---------------------+---------------+-----------------+------------------+-------------------+-----------------------+-------------+
|company_name   |sector    |net_zero_target_year|reduction_pct_claimed|certifications |reddit_post_count|news_article_count|total_media_signals|greenwashing_risk_score|risk_category|
+---------------+----------+--------------------+---------------------+---------------+-----------------+------------------+-------------------+-----------------------+-------------+
|BP             |Energy    |2050                |20.0                 |CDP,SBTi       |100              |25                |125                |1                      |LOW          |
|Barclays       |Finance   |2050                |15.0                 |CDP,SBTi,TCFD  |80               |27                |107                |1                      |LOW          |
|National Grid  |Utilities |2050                |60

### Step 7.4 — Save Processed Output
Save the fully enriched and scored dataset as Parquet for DuckDB analytics.

In [16]:
import pathlib, shutil

# Delete locked file from previous failed Spark write attempt
output_path = "data/processed/greenwashing_merged.parquet"
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Deleted locked file: {output_path}")
    except:
        # If file is still locked, write to a different name
        output_path = "data/processed/greenwashing_merged_v2.parquet"
        print(f"File still locked — writing to: {output_path}")

# Collect Spark DataFrame to Pandas
merged_pd = merged.toPandas()

# Write with pandas/pyarrow — no Hadoop needed
merged_pd.to_parquet(output_path, index=False)
print(f"✅ Saved → {output_path}")
print(f"   Rows    : {len(merged_pd)}")
print(f"   Columns : {list(merged_pd.columns)}")

log_lineage("Spark Processing — Merged + Scored Dataset",
            "data/raw/ (claims + social_signal)",
            len(merged_pd), output_path,
            transformations=[
                "Left join: claims + social_signal on company_name",
                "Derived: has_net_zero_target, high_reduction_claim",
                "Derived: no_certification, high_media_attention, claim_without_cert",
                "Derived: greenwashing_risk_score (0–4)",
                "Derived: risk_category (MINIMAL/LOW/MEDIUM/HIGH)",
                "5 Spark SQL queries executed"
            ])

spark.stop()
print("✅ Spark session closed")

merged_pd[["company_name","sector","reduction_pct_claimed","certifications",
           "total_media_signals","greenwashing_risk_score","risk_category"]]

Deleted locked file: data/processed/greenwashing_merged.parquet
✅ Saved → data/processed/greenwashing_merged.parquet
   Rows    : 10
   Columns : ['company_name', 'company_id', 'sector', 'net_zero_target_year', 'reduction_pct_claimed', 'certifications', 'urls_successful', 'source_url', 'scraped_at', 'reddit_post_count', 'reddit_avg_score', 'reddit_total_comments', 'news_article_count', 'total_media_signals', 'has_net_zero_target', 'high_reduction_claim', 'no_certification', 'high_media_attention', 'claim_without_cert', 'greenwashing_risk_score', 'risk_category', 'processed_at']
[LINEAGE] Spark Processing — Merged + Scored Dataset → 10 records → data/processed/greenwashing_merged.parquet
✅ Spark session closed


,company_name,sector,reduction_pct_claimed,certifications,total_media_signals,greenwashing_risk_score,risk_category
0,BP,Energy,20.0,"CDP,SBTi",125,1,LOW
1,Barclays,Finance,15.0,"CDP,SBTi,TCFD",107,1,LOW
2,Lloyds,Finance,50.0,"CDP,SBTi",94,1,LOW
3,Rio Tinto,Mining,15.0,SBTi,82,1,LOW
4,GSK,Healthcare,80.0,"RE100,SBTi,TCFD",70,1,LOW
5,AstraZeneca,Healthcare,30.0,"CDP,SBTi",84,1,LOW
6,Marks & Spencer,Retail,90.0,"CDP,SBTi",61,1,LOW
7,Vodafone,Telecom,84.0,"CDP,SBTi",84,1,LOW
8,National Grid,Utilities,60.0,"CDP,SBTi",102,1,LOW
9,Sainsburys,Retail,80.0,"CDP,SBTi",89,1,LOW


## Section 8: DuckDB — Analytical Data Warehouse

DuckDB is an in-process analytical database that reads Parquet files directly 
without any data loading step. It is optimised for OLAP (Online Analytical 
Processing) workloads — exactly what we need for our final analysis.

**Why DuckDB after Spark?**
Spark is best for distributed transformation at scale. DuckDB is best for 
interactive analytical SQL on local data. Using both demonstrates the 
appropriate tool for each layer of the pipeline.

We run 6 analytical queries producing the final insights of the pipeline.

In [17]:
import io, os
import pyarrow as pa
import pyarrow.parquet as pq

# Rebuild merged_pd from Parquet files
claims_pd = pd.read_parquet("data/raw/company_claims.parquet")
social_pd = pd.read_parquet("data/raw/social_signal.parquet")

merged_pd = claims_pd.merge(social_pd, on="company_name", how="left")
for col in ["reddit_post_count","reddit_avg_score","reddit_total_comments",
            "news_article_count","total_media_signals"]:
    merged_pd[col] = pd.to_numeric(merged_pd[col], errors="coerce").fillna(0).astype(int)

merged_pd["has_net_zero_target"]     = merged_pd["net_zero_target_year"].notna().astype(int)
merged_pd["high_reduction_claim"]    = (merged_pd["reduction_pct_claimed"] > 30).astype(int)
merged_pd["no_certification"]        = merged_pd["certifications"].isna().astype(int)
merged_pd["high_media_attention"]    = (merged_pd["total_media_signals"] > 50).astype(int)
merged_pd["claim_without_cert"]      = ((merged_pd["reduction_pct_claimed"] > 30) & merged_pd["certifications"].isna()).astype(int)
merged_pd["greenwashing_risk_score"] = (merged_pd["claim_without_cert"] * 2) + merged_pd["no_certification"] + merged_pd["high_media_attention"]
merged_pd["risk_category"]           = merged_pd["greenwashing_risk_score"].apply(
    lambda x: "HIGH" if x >= 3 else ("MEDIUM" if x == 2 else ("LOW" if x == 1 else "MINIMAL")))
merged_pd["processed_at"]            = pd.Timestamp.now().isoformat()

print(f"Merged: {len(merged_pd)} rows, {len(merged_pd.columns)} columns")

# ── Write to a FRESH filename — old file is permanently locked ──
os.makedirs("data/processed", exist_ok=True)
output_path = "data/processed/gw_final.parquet"   # new name, never touched before

buffer = io.BytesIO()
pq.write_table(pa.Table.from_pandas(merged_pd, preserve_index=False), buffer)
with open(output_path, "wb") as f:
    f.write(buffer.getvalue())

print(f"✅ Saved → {output_path}")
print(f"   File size : {os.path.getsize(output_path):,} bytes")

log_lineage("Spark Processing — Merged + Scored Dataset",
            "data/raw/ (claims + social_signal)",
            len(merged_pd), output_path,
            transformations=[
                "Left join: claims + social_signal on company_name",
                "Derived: greenwashing_risk_score (0-4)",
                "Derived: risk_category (MINIMAL/LOW/MEDIUM/HIGH)"
            ])

merged_pd[["company_name","sector","reduction_pct_claimed",
           "certifications","greenwashing_risk_score","risk_category"]]

Merged: 10 rows, 22 columns
✅ Saved → data/processed/gw_final.parquet
   File size : 16,853 bytes
[LINEAGE] Spark Processing — Merged + Scored Dataset → 10 records → data/processed/gw_final.parquet


,company_name,sector,reduction_pct_claimed,certifications,greenwashing_risk_score,risk_category
0,BP,Energy,20.0,"CDP,SBTi",1,LOW
1,Barclays,Finance,15.0,"CDP,SBTi,TCFD",1,LOW
2,Lloyds,Finance,50.0,"CDP,SBTi",1,LOW
3,Rio Tinto,Mining,15.0,SBTi,1,LOW
4,GSK,Healthcare,80.0,"RE100,SBTi,TCFD",1,LOW
5,AstraZeneca,Healthcare,30.0,"CDP,SBTi",1,LOW
6,Marks & Spencer,Retail,90.0,"CDP,SBTi",1,LOW
7,Vodafone,Telecom,84.0,"CDP,SBTi",1,LOW
8,National Grid,Utilities,60.0,"CDP,SBTi",1,LOW
9,Sainsburys,Retail,80.0,"CDP,SBTi",1,LOW


In [18]:
con = duckdb.connect()
con.execute("CREATE VIEW reddit       AS SELECT * FROM read_parquet('data/raw/reddit_posts.parquet')")
con.execute("CREATE VIEW news         AS SELECT * FROM read_parquet('data/raw/news_articles.parquet')")
con.execute("CREATE VIEW uk_co2       AS SELECT * FROM read_parquet('data/raw/owid_co2.parquet')")
con.execute("CREATE VIEW greenwashing AS SELECT * FROM read_parquet('data/processed/gw_final.parquet')")

n = con.execute("SELECT COUNT(*) FROM greenwashing").df().iloc[0,0]
print(f"✅ DuckDB connected — {n} companies in analytical layer")


✅ DuckDB connected — 10 companies in analytical layer


### DuckDB Query 1 — Full Greenwashing Risk Ranking
Complete picture of all companies sorted by risk score.

In [19]:
q1 = con.execute("""
    SELECT company_name, sector,
           COALESCE(CAST(net_zero_target_year AS VARCHAR), 'Not stated')  AS net_zero_target,
           COALESCE(CAST(reduction_pct_claimed AS VARCHAR), 'Not stated') AS reduction_claimed,
           COALESCE(certifications, 'None')                               AS certifications,
           reddit_post_count, news_article_count, total_media_signals,
           greenwashing_risk_score, risk_category
    FROM greenwashing
    ORDER BY greenwashing_risk_score DESC, total_media_signals DESC
""").df()
print("=== GREENWASHING RISK RANKING ===")
q1

=== GREENWASHING RISK RANKING ===


,company_name,sector,net_zero_target,reduction_claimed,certifications,reddit_post_count,news_article_count,total_media_signals,greenwashing_risk_score,risk_category
0,BP,Energy,2050,20.0,"CDP,SBTi",100,25,125,1,LOW
1,Barclays,Finance,2050,15.0,"CDP,SBTi,TCFD",80,27,107,1,LOW
2,National Grid,Utilities,2050,60.0,"CDP,SBTi",76,26,102,1,LOW
3,Lloyds,Finance,2050,50.0,"CDP,SBTi",70,24,94,1,LOW
4,Sainsburys,Retail,2050,80.0,"CDP,SBTi",59,30,89,1,LOW
5,AstraZeneca,Healthcare,2050,30.0,"CDP,SBTi",60,24,84,1,LOW
6,Vodafone,Telecom,2050,84.0,"CDP,SBTi",59,25,84,1,LOW
7,Rio Tinto,Mining,2050,15.0,SBTi,60,22,82,1,LOW
8,GSK,Healthcare,2030,80.0,"RE100,SBTi,TCFD",47,23,70,1,LOW
9,Marks & Spencer,Retail,2050,90.0,"CDP,SBTi",43,18,61,1,LOW


### DuckDB Query 2 — Sector Analysis
Which sectors have the highest average greenwashing risk?

In [20]:
q2 = con.execute("""
    SELECT sector,
           COUNT(*)                                AS companies,
           ROUND(AVG(reduction_pct_claimed), 1)    AS avg_reduction_claimed_pct,
           SUM(no_certification)                   AS uncertified,
           SUM(claim_without_cert)                 AS unverified_claims,
           ROUND(AVG(greenwashing_risk_score), 2)  AS avg_risk_score,
           MAX(risk_category)                      AS max_risk_level,
           SUM(total_media_signals)                AS total_media_exposure
    FROM greenwashing
    GROUP BY sector
    ORDER BY avg_risk_score DESC
""").df()
print("=== SECTOR GREENWASHING RISK ===")
q2

=== SECTOR GREENWASHING RISK ===


,sector,companies,avg_reduction_claimed_pct,uncertified,unverified_claims,avg_risk_score,max_risk_level,total_media_exposure
0,Mining,1,15.0,0.0,0.0,1.0,LOW,82.0
1,Finance,2,32.5,0.0,0.0,1.0,LOW,201.0
2,Telecom,1,84.0,0.0,0.0,1.0,LOW,84.0
3,Utilities,1,60.0,0.0,0.0,1.0,LOW,102.0
4,Energy,1,20.0,0.0,0.0,1.0,LOW,125.0
5,Healthcare,2,55.0,0.0,0.0,1.0,LOW,154.0
6,Retail,2,85.0,0.0,0.0,1.0,LOW,150.0


### DuckDB Query 3 — Reddit Sentiment Deep Dive
What are the most upvoted Reddit posts about each company's environmental record?
High score = community found this significant.

In [21]:
q3 = con.execute("""
    SELECT company_name, title, subreddit, score, num_comments, upvote_ratio, created_utc
    FROM reddit
    WHERE score > 1000
    ORDER BY score DESC
    LIMIT 20
""").df()
print("=== TOP REDDIT POSTS BY UPVOTE SCORE ===")
q3

=== TOP REDDIT POSTS BY UPVOTE SCORE ===


,company_name,title,subreddit,score,num_comments,upvote_ratio,created_utc
0,Lloyds,Michael J Fox and Cristopher Lloyd reception a...,nextfuckinglevel,110950,2876,0.93,2022-10-09T22:39:35
1,AstraZeneca,"Sarah Gilbert, co-designer of the Oxford/Astra...",pics,87626,1375,0.79,2021-06-28T18:28:44
2,Barclays,The Truck elevators and turntable underneath t...,nextfuckinglevel,84613,913,0.97,2020-08-22T20:37:19
3,AstraZeneca,Covid eight times more likely to cause rare br...,Coronavirus,56064,1659,0.84,2021-04-15T11:34:54
4,Rio Tinto,Rio Tinto legally destroy Native Australian Ab...,worldnews,50791,1824,0.93,2020-05-27T22:27:39
5,AstraZeneca,Researchers have debunked a popular anti-vacci...,science,44053,3082,0.86,2021-07-30T05:51:07
6,BP,"BP, Unilever, and HSBC have failed to properly...",worldnews,43224,1121,0.94,2023-01-02T07:40:07
7,Vodafone,British travellers rage as Vodafone brings bac...,LeopardsAteMyFace,42694,2724,0.91,2021-11-08T12:05:56
8,Sainsburys,Bought a skeleton from Sainsbury’s. I regret n...,CasualUK,41468,855,0.96,2025-09-26T19:13:29
9,BP,My father’s 6th attempt to outsmart his geriat...,funny,39994,1165,0.94,2023-03-10T00:41:13


### DuckDB Query 4 — News Coverage Analysis
Which companies receive the most Guardian journalism scrutiny, and in which sections?

In [22]:
q4 = con.execute("""
    SELECT company_name,
           COUNT(*)                                    AS total_articles,
           COUNT(DISTINCT section)                     AS sections_covered,
           MIN(published_at)                           AS earliest_article,
           MAX(published_at)                           AS latest_article
    FROM news
    GROUP BY company_name
    ORDER BY total_articles DESC
""").df()
print("=== GUARDIAN COVERAGE BY COMPANY ===")
q4

=== GUARDIAN COVERAGE BY COMPANY ===


,company_name,total_articles,sections_covered,earliest_article,latest_article
0,Sainsburys,30,9,2025-02-18 00:01:51+00:00,2026-03-04 11:00:18+00:00
1,Barclays,27,7,2025-07-01 01:08:56+01:00,2026-03-04 11:00:18+00:00
2,National Grid,26,8,2025-03-23 22:51:47+00:00,2026-03-04 11:00:18+00:00
3,BP,25,7,2025-10-15 02:14:32+01:00,2026-03-04 11:00:18+00:00
4,Vodafone,25,6,2025-10-06 20:30:09+01:00,2026-03-04 11:00:18+00:00
5,Lloyds,24,5,2025-10-07 15:00:04+01:00,2026-03-04 11:00:18+00:00
6,AstraZeneca,24,5,2025-09-12 19:29:52+01:00,2026-03-04 11:00:18+00:00
7,GSK,23,5,2025-09-29 18:48:51+01:00,2026-03-04 11:00:18+00:00
8,Rio Tinto,22,6,2024-11-20 13:10:09+00:00,2026-03-04 11:00:18+00:00
9,Marks & Spencer,18,7,2025-04-25 15:36:08+01:00,2026-02-27 10:36:13+00:00


### DuckDB Query 5 — Claim Credibility Assessment
For companies that made specific claims, how do those claims compare 
to their certification status and media scrutiny?

In [23]:
q5 = con.execute("""
    SELECT company_name, sector,
           reduction_pct_claimed,
           certifications,
           CASE 
               WHEN certifications IS NOT NULL AND reduction_pct_claimed IS NOT NULL 
                   THEN 'Verified claim'
               WHEN certifications IS NULL AND reduction_pct_claimed IS NOT NULL 
                   THEN 'Unverified claim — HIGH RISK'
               WHEN certifications IS NOT NULL AND reduction_pct_claimed IS NULL 
                   THEN 'Certified but no specific claim'
               ELSE 'No claim and no certification'
           END AS claim_credibility,
           total_media_signals,
           risk_category
    FROM greenwashing
    ORDER BY greenwashing_risk_score DESC
""").df()
print("=== CLAIM CREDIBILITY ASSESSMENT ===")
q5

=== CLAIM CREDIBILITY ASSESSMENT ===


,company_name,sector,reduction_pct_claimed,certifications,claim_credibility,total_media_signals,risk_category
0,BP,Energy,20.0,"CDP,SBTi",Verified claim,125,LOW
1,Barclays,Finance,15.0,"CDP,SBTi,TCFD",Verified claim,107,LOW
2,Lloyds,Finance,50.0,"CDP,SBTi",Verified claim,94,LOW
3,Rio Tinto,Mining,15.0,SBTi,Verified claim,82,LOW
4,GSK,Healthcare,80.0,"RE100,SBTi,TCFD",Verified claim,70,LOW
5,AstraZeneca,Healthcare,30.0,"CDP,SBTi",Verified claim,84,LOW
6,Marks & Spencer,Retail,90.0,"CDP,SBTi",Verified claim,61,LOW
7,Vodafone,Telecom,84.0,"CDP,SBTi",Verified claim,84,LOW
8,National Grid,Utilities,60.0,"CDP,SBTi",Verified claim,102,LOW
9,Sainsburys,Retail,80.0,"CDP,SBTi",Verified claim,89,LOW


### DuckDB Query 6 — UK National CO2 Context
How have UK national emissions trended since 2015? 
This provides macroeconomic context for company-level claims.

In [24]:
q6 = con.execute("""
    SELECT year,
           ROUND(co2, 2)            AS total_co2_MtCO2,
           ROUND(total_ghg, 2)      AS total_ghg,
           ROUND(co2_per_capita, 2) AS co2_per_capita,
           ROUND(coal_co2, 2)       AS coal_co2,
           ROUND(oil_co2, 2)        AS oil_co2,
           ROUND(gas_co2, 2)        AS gas_co2,
           ROUND(100.0 * (co2 - LAG(co2) OVER (ORDER BY year)) / 
                 LAG(co2) OVER (ORDER BY year), 1) AS yoy_change_pct
    FROM uk_co2
    WHERE year >= 2015
    ORDER BY year DESC
""").df()
print("=== UK NATIONAL CO2 TREND WITH YoY CHANGE ===")
q6

=== UK NATIONAL CO2 TREND WITH YoY CHANGE ===


,year,total_co2_MtCO2,total_ghg,co2_per_capita,coal_co2,oil_co2,gas_co2,yoy_change_pct
0,2024,312.91,390.28,4.53,16.95,159.60,126.62,1.7
1,2023,307.83,386.61,4.48,17.91,154.68,125.77,-1.1
2,2022,311.12,392.13,4.56,19.98,146.33,135.05,-9.1
3,2021,342.37,425.77,5.06,23.76,149.63,158.50,4.9
4,2020,326.26,411.77,4.84,22.81,143.98,149.42,-10.6
5,2019,364.75,453.99,5.44,24.51,169.00,159.32,-3.9
6,2018,379.73,475.27,5.69,33.28,173.09,161.49,-2.0
7,2017,387.37,477.85,5.84,39.13,174.85,161.70,-3.0
8,2016,399.43,494.14,6.06,47.83,173.93,165.74,-5.5
9,2015,422.46,518.77,6.46,91.55,170.77,147.90,NaN


### Save Analytical Outputs

In [25]:
q1.to_parquet("data/processed/risk_ranking.parquet",    index=False)
q2.to_parquet("data/processed/sector_analysis.parquet", index=False)
q5.to_parquet("data/processed/claim_credibility.parquet", index=False)
q6.to_parquet("data/processed/uk_co2_trend.parquet",    index=False)

log_lineage("DuckDB Analytical Layer",
            "data/processed/greenwashing_merged.parquet",
            len(q1), "data/processed/risk_ranking.parquet",
            transformations=["Query 1: Full risk ranking",
                             "Query 2: Sector aggregation",
                             "Query 3: Reddit top posts by score",
                             "Query 4: Guardian coverage by company",
                             "Query 5: Claim credibility assessment",
                             "Query 6: UK CO2 trend with YoY change"])
print("✅ All analytical outputs saved")

[LINEAGE] DuckDB Analytical Layer → 10 records → data/processed/risk_ranking.parquet
✅ All analytical outputs saved


## Section 9: Data Lineage Summary

Data lineage is the full audit trail of where every record came from, 
what transformations were applied, and where it ended up. 
This is a core data engineering best practice for reproducibility and compliance.

In [26]:
lineage = []
with open("data/lineage_log.jsonl") as f:
    for line in f:
        lineage.append(json.loads(line))

lineage_df = pd.DataFrame(lineage)
print("=== FULL DATA LINEAGE LOG ===")
lineage_df[["source","record_count","output_path","extracted_at"]]

=== FULL DATA LINEAGE LOG ===


,source,record_count,output_path,extracted_at
0,FTSE100 Multi-URL Web Scraping (4 URLs per com...,10,data/raw/company_claims.parquet,2026-03-04T11:33:07.835750
1,The Guardian Open API,244,data/raw/news_articles.parquet,2026-03-04T11:34:05.358524
2,Reddit Public JSON API,654,data/raw/reddit_posts.parquet,2026-03-04T11:42:00.530793
3,Our World in Data — UK CO2 Dataset,25,data/raw/owid_co2.parquet,2026-03-04T11:42:04.199190
4,NetworkX Graph — Company ESG Relationships,21,data/processed/company_graph.gexf,2026-03-04T11:42:04.236895
5,Social Signal Table (Reddit + Guardian aggrega...,10,data/raw/social_signal.parquet,2026-03-04T11:42:04.308469
6,Spark Processing — Merged + Scored Dataset,10,data/processed/greenwashing_merged.parquet,2026-03-04T11:42:28.119783
7,Spark Processing — Merged + Scored Dataset,10,data/processed/gw_final.parquet,2026-03-04T11:42:29.121572
8,DuckDB Analytical Layer,10,data/processed/risk_ranking.parquet,2026-03-04T11:42:29.803280


In [27]:
print("=" * 65)
print("PIPELINE SUMMARY — MSIN0166 Greenwashing Detection")
print("=" * 65)
print(f"Total sources    : {len(lineage_df)}")
print(f"Total records    : {lineage_df['record_count'].sum():,}")
print()
for _, row in lineage_df.iterrows():
    print(f"  {row['source']}")
    print(f"    {row['record_count']} records → {row['output_path']}")
print()
print("Storage Formats:")
print("  MongoDB  — raw unstructured text (NoSQL document store)")
print("  Parquet  — structured tabular data (columnar, compressed)")
print("  GEXF     — company relationship graph (graph database)")
print("  JSONL    — lineage log (append-only audit trail)")
print()
print("Processing Stack:")
print("  Apache Spark — distributed join, transformation, Spark SQL")
print("  DuckDB       — in-process OLAP analytical queries")
print("  NetworkX     — graph database relationship modelling")
print("=" * 65)

PIPELINE SUMMARY — MSIN0166 Greenwashing Detection
Total sources    : 9
Total records    : 994

  FTSE100 Multi-URL Web Scraping (4 URLs per company)
    10 records → data/raw/company_claims.parquet
  The Guardian Open API
    244 records → data/raw/news_articles.parquet
  Reddit Public JSON API
    654 records → data/raw/reddit_posts.parquet
  Our World in Data — UK CO2 Dataset
    25 records → data/raw/owid_co2.parquet
  NetworkX Graph — Company ESG Relationships
    21 records → data/processed/company_graph.gexf
  Social Signal Table (Reddit + Guardian aggregated)
    10 records → data/raw/social_signal.parquet
  Spark Processing — Merged + Scored Dataset
    10 records → data/processed/greenwashing_merged.parquet
  Spark Processing — Merged + Scored Dataset
    10 records → data/processed/gw_final.parquet
  DuckDB Analytical Layer
    10 records → data/processed/risk_ranking.parquet

Storage Formats:
  MongoDB  — raw unstructured text (NoSQL document store)
  Parquet  — structured 